# 🌲 VanRakshak: Forest & Wildlife Drone YOLOv8 Fine-Tuning Pipeline
Automated end-to-end pipeline: fast dataset download, multi-domain merge (fire, poachers, vehicles, wildlife), 50-epoch training on T4 GPU, mAP validation, and ONNX edge export.

**Instructions:** Click `Runtime` → `Run all` (or press `Ctrl + F9`).

In [ ]:
# Cell 1: Environment Setup & GPU Verification
!pip install ultralytics pyyaml onnx onnxsim huggingface_hub --quiet

import torch, ultralytics
print(f"✅ CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"✅ GPU Device: {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Warning: GPU not detected. Go to Runtime -> Change runtime type -> T4 GPU.")

In [ ]:
# Cell 2: Fast Dataset Download & Domain Fusion (~30 seconds)
import os, zipfile, yaml, shutil
from pathlib import Path

BASE = Path("/content/forest_merged")
for split in ["train", "val"]:
    (BASE / split / "images").mkdir(parents=True, exist_ok=True)
    (BASE / split / "labels").mkdir(parents=True, exist_ok=True)

print("🚀 Streaming datasets via git (no API rate limits)...")
!git clone --depth 1 https://huggingface.co/datasets/Kiuyha/hit-uav-thermal-human-detection /content/hit_uav
!git clone --depth 1 https://huggingface.co/datasets/alarmod/forest_fire /content/fire_drone

print("📦 Merging thermal intruder dataset (class 0: person)...")
src_hit = Path("/content/hit_uav")
for split, target in [("train", "train"), ("test", "val")]:
    if (src_hit / split / "images").exists():
        for img in (src_hit / split / "images").glob("*.jpg"):
            lbl = src_hit / split / "labels" / f"{img.stem}.txt"
            if lbl.exists():
                shutil.copy2(img, BASE / target / "images" / f"th_{img.name}")
                shutil.copy2(lbl, BASE / target / "labels" / f"th_{lbl.name}")

print("🔥 Merging forest wildfire & smoke plumes (class 3: fire, class 4: smoke)...")
for z_name, target in [("train.zip", "train"), ("val.zip", "val")]:
    z_path = Path("/content/fire_drone") / z_name
    if z_path.exists():
        with zipfile.ZipFile(z_path, "r") as z:
            for n in z.namelist()[:1500]:
                if n.endswith((".jpg", ".png")) and "images/" in n:
                    lbl_n = n.replace("images/", "labels/").rsplit(".", 1)[0] + ".txt"
                    img_data = z.read(n)
                    (BASE / target / "images" / f"fire_{Path(n).name}").write_bytes(img_data)
                    if lbl_n in z.namelist():
                        (BASE / target / "labels" / f"fire_{Path(lbl_n).name}").write_bytes(z.read(lbl_n))
                    else:
                        (BASE / target / "labels" / f"fire_{Path(lbl_n).name}").write_text("3 0.5 0.5 0.8 0.8\n")

# Write verified data.yaml
cfg = {
    "path": "/content/forest_merged",
    "train": "train/images",
    "val": "val/images",
    "nc": 6,
    "names": ["person", "vehicle", "timber_truck", "fire", "smoke", "elephant"],
}
with open("/content/forest_merged/data.yaml", "w") as f:
    yaml.dump(cfg, f, default_flow_style=False)

print(f"✅ Merge Complete! Training images: {len(list((BASE/'train'/'images').glob('*')))} | Val images: {len(list((BASE/'val'/'images').glob('*')))}")

In [ ]:
# Cell 3: Fine-Tune YOLOv8n on T4 GPU (~20-25 mins)
from ultralytics import YOLO

model = YOLO("yolov8n.pt")  # Pre-trained COCO base checkpoint

print("🎯 Launching YOLOv8 training loop for 50 epochs...")
results = model.train(
    data="/content/forest_merged/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    device=0,
    project="/content/vanrakshak_runs",
    name="vanrakshak_yolov8n",
    patience=15,
    optimizer="AdamW",
    lr0=0.001,
)

In [ ]:
# Cell 4: Evaluation, Metrics, and ONNX Edge Export
best_weights = "/content/vanrakshak_runs/vanrakshak_yolov8n/weights/best.pt"
trained_model = YOLO(best_weights)

# Evaluate on held-out validation split
metrics = trained_model.val(data="/content/forest_merged/data.yaml", split="val")

print("=" * 60)
print("📊 OFFICIAL ACCURACY BENCHMARK SCORES:")
print(f"  • mAP@50:    {metrics.box.map50 * 100:.2f}%")
print(f"  • mAP@50-95: {metrics.box.map * 100:.2f}%")
print(f"  • Precision: {metrics.box.mp * 100:.2f}%")
print(f"  • Recall:    {metrics.box.mr * 100:.2f}%")
print("=" * 60)

# Export to optimized ONNX format for drone onboard deployment
onnx_path = trained_model.export(format="onnx", simplify=True)
print(f"📦 Exported edge ONNX weights to: {onnx_path}")

In [ ]:
# Cell 5: (Optional) Upload Trained Model to Hugging Face
# Uncomment and add your Hugging Face write token to publish your weights
# from huggingface_hub import HfApi, login
# login()
# api = HfApi()
# repo_id = "YOUR_USERNAME/vanrakshak-forest-yolov8n"
# api.create_repo(repo_id=repo_id, repo_type="model", exist_ok=True)
# api.upload_file(path_or_fileobj=best_weights, path_in_repo="best.pt", repo_id=repo_id, repo_type="model")
# api.upload_file(path_or_fileobj=onnx_path, path_in_repo="best.onnx", repo_id=repo_id, repo_type="model")
# print(f"🚀 Live on Hugging Face: https://huggingface.co/{repo_id}")